In [3]:
import json
import numpy as np
import pprint
from transformers import AutoTokenizer

def get_token_stats(jsonl_path, model_name, prompt_key='prompt', response_key='response'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    lengths = []
    p_lens = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)
            
            full_text = data.get(prompt_key, "") + "\n" + data.get(response_key, "")
            tokens = tokenizer.encode(full_text, add_special_tokens=False)
            lengths.append(len(tokens))

            p_lens.append(len(tokenizer.encode(data.get(prompt_key, ""), add_special_tokens=False)))
            
    if not lengths:
        return {}

    # Tính toán các chỉ số và gom vào dictionary
    full_text_stats = {
        'count': len(lengths),
        'min': int(np.min(lengths)),
        'max': int(np.max(lengths)),
        'mean': float(np.mean(lengths)),
        'median': float(np.median(lengths)),
        'p50': float(np.percentile(lengths, 50)),
        'p90': float(np.percentile(lengths, 90)),
        'p95': float(np.percentile(lengths, 95)),
        'p99': float(np.percentile(lengths, 99))
    }
    prompt_stats = {
        'count': len(p_lens),
        'min': int(np.min(p_lens)),
        'max': int(np.max(p_lens)),
        'mean': float(np.mean(p_lens)),
        'median': float(np.median(p_lens)),
        'p50': float(np.percentile(p_lens, 50)),
        'p90': float(np.percentile(p_lens, 90)),
        'p95': float(np.percentile(p_lens, 95)),
        'p99': float(np.percentile(p_lens, 99))
    }
    stats = {
        'full_text': full_text_stats,
        'prompt': prompt_stats
    }
    
    return stats


In [4]:
FILE_PATH = "./data/dpo/Qwen/Qwen2.5-Math-1.5B-Instruct/generated_train.jsonl"  
MODEL_NAME = "Qwen/Qwen2.5-0.5B" 

stats_dict = get_token_stats(FILE_PATH, MODEL_NAME, prompt_key='prompt', response_key='generated_text')

pprint.pprint(stats_dict, sort_dicts=False)

{'full_text': {'count': 79751,
               'min': 49,
               'max': 13592,
               'mean': 614.1741545560557,
               'median': 520.0,
               'p50': 520.0,
               'p90': 1105.0,
               'p95': 1274.0,
               'p99': 1686.0},
 'prompt': {'count': 79751,
            'min': 14,
            'max': 13592,
            'mean': 188.87844666524558,
            'median': 100.0,
            'p50': 100.0,
            'p90': 491.0,
            'p95': 652.0,
            'p99': 1004.0}}


In [ ]:
from datasets import load_dataset
import json

def process_and_save_dataset():
    dataset_name = "Minsang/TSD-KD-Qwen2.5-1.5B-Instruct-Gen"
    output_filename = "data_output.jsonl"

    print(f"Đang tải dữ liệu từ {dataset_name}...")
    
    # Tải dataset (thường mặc định sẽ tải split 'train')
    # Lưu ý: Tuỳ thuộc vào cấu trúc dataset, có thể bạn cần thay đổi split='train' nếu cần
    dataset = load_dataset(dataset_name, split="train")
    
    print(f"Đã tải xong {len(dataset)} dòng dữ liệu. Đang tiến hành ghi ra file {output_filename}...")

    # Mở file để ghi dưới dạng JSONL
    with open(output_filename, "w", encoding="utf-8") as outfile:
        for row in dataset:
            # Lấy dữ liệu từ các cột tương ứng, gán giá trị rỗng nếu không tồn tại
            instruction = row.get("instruction", "")
            response = row.get("response", "")
            
            # Tạo dictionary với định dạng mới
            new_row = {
                "prompt": instruction,
                "generated_text": response
            }
            
            # Chuyển đổi thành chuỗi JSON và ghi vào file, thêm ký tự xuống dòng
            json_line = json.dumps(new_row, ensure_ascii=False)
            outfile.write(json_line + "\n")

    print("Hoàn tất quá trình lưu file!")

if __name__ == "__main__":
    process_and_save_dataset()

In [6]:
from datasets import load_dataset
import json
import os

def process_and_save_dataset(dataset_name, output_filename):
    os.makedirs(os.path.dirname(output_filename), exist_ok=True)
    print(f"Đang tải dữ liệu từ {dataset_name}...")
    
    dataset = load_dataset(dataset_name, split="train")
    
    print(f"Đã tải xong {len(dataset)} dòng dữ liệu. Đang tiến hành ghi ra file {output_filename}...")

    with open(output_filename, "w", encoding="utf-8") as outfile:
        for row in dataset:
            instruction = row.get("instruction", "")
            response = row.get("response", "")
            
            new_row = {
                "prompt": instruction,
                "generated_text": response
            }

            json_line = json.dumps(new_row, ensure_ascii=False)
            outfile.write(json_line + "\n")

    print("Hoàn tất quá trình lưu file!")


In [8]:
process_and_save_dataset("Minsang/TSD-KD-Qwen2.5-1.5B-Instruct-Gen", "data/dpo/Qwen/Qwen2.5-1.5B-Instruct/generated_train.jsonl")


Đang tải dữ liệu từ Minsang/TSD-KD-Qwen2.5-1.5B-Instruct-Gen...


Đã tải xong 79751 dòng dữ liệu. Đang tiến hành ghi ra file data/dpo/Qwen/Qwen2.5-1.5B-Instruct/generated_train.jsonl...
Hoàn tất quá trình lưu file!


In [9]:
process_and_save_dataset("Minsang/TSD-KD-Qwen2.5-14B-Instruct-Gen", "data/dpo/Qwen/Qwen2.5-14B-Instruct/generated_train.jsonl")


Đang tải dữ liệu từ Minsang/TSD-KD-Qwen2.5-14B-Instruct-Gen...


Generating train split: 100%|██████████| 79751/79751 [00:00<00:00, 85866.20 examples/s]


Đã tải xong 79751 dòng dữ liệu. Đang tiến hành ghi ra file data/dpo/Qwen/Qwen2.5-14B-Instruct/generated_train.jsonl...
Hoàn tất quá trình lưu file!
